# Loading Team State from a File

In [1]:
import asyncio
from autogen_core import CancellationToken
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [2]:
import json

with open('team_state.json', 'r') as f:
    team_state = json.load(f)

team_state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'Writer_1': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'Write a poem about the sea in 3 lines',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': 'Vast waves dance with grace,  \nwhispering secrets untold,  \nblue embrace of dreams.',
       'thought': None,
       'source': 'Writer_1',
       'type': 'AssistantMessage'}]}},
   'message_buffer': [{'id': '546ecd78-3e3b-4177-8324-572abe4e29a9',
     'source': 'Writer_2',
     'models_usage': {'prompt_tokens': 66, 'completion_tokens': 20},
     'metadata': {},
     'created_at': '2026-06-27T03:59:09.256680Z',
     'content': 'Endless blue whispers,  \nmoonlit waves cradle the stars,  \nsecrets in the deep.',
     'type': 'TextMessage'}]},
  'Writer_2': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_st

In [3]:
from autogen_agentchat.conditions import MaxMessageTermination

agent_1 = AssistantAgent(
    name='Writer_1',
    model_client=model_client,
    system_message="You are a helpful assistant. Give the output in less than 30 words",
)

agent_2 = AssistantAgent(
    name='Writer_2',
    model_client=model_client,
    system_message="You are a helpful assistant.Give the output in less than 30 words",
)

terminationCondition = MaxMessageTermination(max_messages=3)

new_agent_team = RoundRobinGroupChat(participants=[agent_1, agent_2],termination_condition=terminationCondition)

In [4]:
await new_agent_team.load_state(team_state)

In [5]:
from autogen_agentchat.ui import Console

stream = new_agent_team.run_stream(task="What was the last line of Poem you wrote?")

await Console(stream)

---------- TextMessage (user) ----------
What was the last line of Poem you wrote?
---------- TextMessage (Writer_1) ----------
"blue embrace of dreams."
---------- TextMessage (Writer_2) ----------
The last line of my poem was "secrets in the deep."


TaskResult(messages=[TextMessage(id='87dfa3ad-c2b9-416c-b301-efd8f8f45c7b', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 27, 4, 0, 8, 727385, tzinfo=datetime.timezone.utc), content='What was the last line of Poem you wrote?', type='TextMessage'), TextMessage(id='e46d4205-5e4c-4426-817d-b62220c578f4', source='Writer_1', models_usage=RequestUsage(prompt_tokens=106, completion_tokens=6), metadata={}, created_at=datetime.datetime(2026, 6, 27, 4, 0, 10, 357994, tzinfo=datetime.timezone.utc), content='"blue embrace of dreams."', type='TextMessage'), TextMessage(id='fe0ef056-5061-4f42-885f-1bdbf061a23f', source='Writer_2', models_usage=RequestUsage(prompt_tokens=120, completion_tokens=14), metadata={}, created_at=datetime.datetime(2026, 6, 27, 4, 0, 11, 614267, tzinfo=datetime.timezone.utc), content='The last line of my poem was "secrets in the deep."', type='TextMessage')], stop_reason='Maximum number of messages 3 reached, current message count: 3')